# Part-of-Speech Tagging with BERT

In this notebook, we'll implement a Part-of-Speech (POS) tagger by fine-tuning a pre-trained BERT model. We'll compare its performance with previous models (baseline, MLP, RNN, CNN) and analyze the results.


In [ ]:
# !pip install conllu
# !pip install gensim

## 1. Libraries Import


In [ ]:
import os
import urllib.request
import zipfile
import conllu
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Dense, Input, Dropout, Layer
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical
from collections import Counter, defaultdict
from sklearn.metrics import precision_score, recall_score, f1_score, precision_recall_curve, auc, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Transformers imports
from transformers import TFBertModel, BertTokenizer, BertTokenizerFast, BertConfig
from transformers import create_optimizer
from transformers import logging as transformers_logging

# Set TensorFlow to grow GPU memory allocation
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("GPU is available and memory growth is set.")
    except RuntimeError as e:
        print(e)
else:
    print("No GPU detected. Running on CPU.")

# Set transformers logging level
transformers_logging.set_verbosity_error()

# Define a custom loss function that ignores -100 labels
def masked_sparse_categorical_crossentropy(y_true, y_pred):
    # Create a mask for labels != -100
    mask = tf.not_equal(y_true, -100)

    # Get the number of valid labels
    num_valid = tf.reduce_sum(tf.cast(mask, tf.float32))

    # Apply the mask to the labels and predictions
    y_true_masked = tf.boolean_mask(y_true, mask)
    y_pred_masked = tf.boolean_mask(y_pred, mask)

    # Compute the loss on the masked values
    loss = tf.keras.losses.sparse_categorical_crossentropy(
        y_true_masked, y_pred_masked, from_logits=True
    )

    # Return the mean loss over valid labels
    return tf.reduce_sum(loss) / (num_valid + tf.keras.backend.epsilon())

# Define a custom layer to handle casting
class CastLayer(Layer):
    def __init__(self, dtype=tf.int32, **kwargs):
        super(CastLayer, self).__init__(**kwargs)
        self._cast_dtype = dtype

    def call(self, inputs):
        return tf.cast(inputs, self._cast_dtype)

    def get_config(self):
        config = super(CastLayer, self).get_config()
        # Store the dtype as a string representation
        if hasattr(self._cast_dtype, 'name'):
            dtype_str = self._cast_dtype.name
        else:
            dtype_str = str(self._cast_dtype)
        config.update({"dtype_str": dtype_str})
        return config

    @classmethod
    def from_config(cls, config):
        # Convert the string representation back to a TensorFlow dtype
        dtype_str = config.pop('dtype_str', 'int32')
        # Default to int32 if not specified
        dtype = tf.as_dtype(dtype_str)
        return cls(dtype=dtype, **config)

# Define BERT model name constant
BERT_MODEL_NAME = "bert-base-cased"  # We use cased model since case is important for POS tagging

# Define a custom layer to wrap the BERT model
class BertLayer(Layer):
    def __init__(self, bert_model, model_name=BERT_MODEL_NAME, **kwargs):
        super(BertLayer, self).__init__(**kwargs)
        self.bert_model = bert_model
        self.model_name = model_name

    def call(self, inputs):
        input_ids, attention_mask, token_type_ids = inputs
        bert_outputs = self.bert_model(
            input_ids=input_ids,
            attention_mask=tf.cast(attention_mask, tf.int32),
            token_type_ids=token_type_ids
        )
        # Return only the last hidden state
        return bert_outputs.last_hidden_state

    def compute_output_shape(self, input_shape):
        # Return the expected output shape for the last hidden state only
        return (None, MAX_SEQUENCE_LENGTH, 768)

    def get_config(self):
        config = super(BertLayer, self).get_config()
        config.update({"model_name": self.model_name})
        return config

    @classmethod
    def from_config(cls, config):
        model_name = config.pop("model_name", BERT_MODEL_NAME)
        bert_model = TFBertModel.from_pretrained(model_name)
        return cls(bert_model=bert_model, model_name=model_name, **config)


## 2. Data Loading and Preprocessing


In [ ]:
# Constants
LANGUAGE = "English"
TREEBANK = "UD_English-EWT"
TREEBANK_URL = "https://github.com/UniversalDependencies/UD_English-EWT/archive/master.zip"
MAX_SEQUENCE_LENGTH = 128  # Maximum sentence length for BERT
BATCH_SIZE = 16
EPOCHS = 30
LEARNING_RATE = 2e-5


In [ ]:
# Function to download and extract the treebank
def download_treebank(url, treebank_name):
    zip_path = f"{treebank_name}.zip"
    if not os.path.exists(zip_path):
        print(f"Downloading {treebank_name}...")
        urllib.request.urlretrieve(url, zip_path)

    extract_dir = f"{treebank_name}-data"
    if not os.path.exists(extract_dir):
        print(f"Extracting {treebank_name}...")
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_dir)

    return extract_dir

# Function to load and parse the CoNLL-U files
def load_conllu_data(treebank_dir, treebank_name):
    base_dir = os.path.join(treebank_dir, f"{treebank_name}-master")
    train_file = None
    dev_file = None
    test_file = None

    for file in os.listdir(base_dir):
        if file.endswith(".conllu"):
            if "train" in file:
                train_file = os.path.join(base_dir, file)
            elif "dev" in file:
                dev_file = os.path.join(base_dir, file)
            elif "test" in file:
                test_file = os.path.join(base_dir, file)

    train_data = []
    dev_data = []
    test_data = []

    if train_file:
        with open(train_file, "r", encoding="utf-8") as f:
            train_data = conllu.parse(f.read())

    if dev_file:
        with open(dev_file, "r", encoding="utf-8") as f:
            dev_data = conllu.parse(f.read())

    if test_file:
        with open(test_file, "r", encoding="utf-8") as f:
            test_data = conllu.parse(f.read())

    return train_data, dev_data, test_data

# Function to extract sentences and POS tags from the parsed data
def extract_sentences_and_tags(data):
    sentences = []
    pos_tags = []

    for sentence in data:
        words = []
        tags = []

        for token in sentence:
            if token["upos"] != "_":
                words.append(token["form"])  # Keep original case for BERT
                tags.append(token["upos"])

        if words:
            sentences.append(words)
            pos_tags.append(tags)

    return sentences, pos_tags


In [ ]:
# Download and load data
print("Loading data...")
treebank_dir = download_treebank(TREEBANK_URL, TREEBANK)
train_data, dev_data, test_data = load_conllu_data(treebank_dir, TREEBANK)

# Extract sentences and tags
train_sentences, train_pos_tags = extract_sentences_and_tags(train_data)
dev_sentences, dev_pos_tags = extract_sentences_and_tags(dev_data)
test_sentences, test_pos_tags = extract_sentences_and_tags(test_data)


In [ ]:
# Calculate dataset statistics
def calculate_dataset_stats(sentences, tags):
    num_sentences = len(sentences)
    num_words = sum(len(s) for s in sentences)
    avg_sentence_length = num_words / num_sentences if num_sentences > 0 else 0

    # Calculate vocabulary size
    vocab = set()
    for sentence in sentences:
        vocab.update(sentence)
    vocab_size = len(vocab)

    # Calculate tag distribution
    tag_counter = Counter()
    for tag_seq in tags:
        tag_counter.update(tag_seq)

    return {
        "num_sentences": num_sentences,
        "num_words": num_words,
        "avg_sentence_length": avg_sentence_length,
        "vocab_size": vocab_size,
        "tag_distribution": tag_counter
    }

# Calculate and display dataset statistics
print("Calculating dataset statistics...")
train_stats = calculate_dataset_stats(train_sentences, train_pos_tags)
dev_stats = calculate_dataset_stats(dev_sentences, dev_pos_tags)
test_stats = calculate_dataset_stats(test_sentences, test_pos_tags)

print(f"Dataset Statistics for {LANGUAGE} ({TREEBANK}):\n")
print("Training Set:")
print(f"  Number of sentences: {train_stats['num_sentences']}")
print(f"  Number of words: {train_stats['num_words']}")
print(f"  Average sentence length: {train_stats['avg_sentence_length']:.2f}")
print(f"  Vocabulary size: {train_stats['vocab_size']}")
print("\nDevelopment Set:")
print(f"  Number of sentences: {dev_stats['num_sentences']}")
print(f"  Number of words: {dev_stats['num_words']}")
print(f"  Average sentence length: {dev_stats['avg_sentence_length']:.2f}")
print("\nTest Set:")
print(f"  Number of sentences: {test_stats['num_sentences']}")
print(f"  Number of words: {test_stats['num_words']}")
print(f"  Average sentence length: {test_stats['avg_sentence_length']:.2f}")

# Display tag distribution
print("\nPOS Tag Distribution (Training Set):")
for tag, count in sorted(train_stats['tag_distribution'].items(), key=lambda x: x[1], reverse=True):
    print(f"  {tag}: {count} ({count/train_stats['num_words']*100:.2f}%)")


## 3. Create Tag Mappings


In [ ]:
# Create tag mappings
def create_tag_mappings(train_tags):
    # Create tag-to-index mapping
    tag_to_idx = {}
    for tag_seq in train_tags:
        for tag in tag_seq:
            if tag not in tag_to_idx:
                tag_to_idx[tag] = len(tag_to_idx)

    # Create index-to-tag mapping for later use
    idx_to_tag = {idx: tag for tag, idx in tag_to_idx.items()}

    return tag_to_idx, idx_to_tag

# Create mappings
print("Creating tag mappings...")
tag_to_idx, idx_to_tag = create_tag_mappings(train_pos_tags)
print(f"Number of POS tags: {len(tag_to_idx)}")
print(f"POS tags: {list(tag_to_idx.keys())}")


## 4. Baseline Tagger


In [ ]:
# Baseline Tagger
class BaselineTagger:
    def __init__(self):
        self.word_to_tag = {}
        self.most_common_tag = None

    def train(self, sentences, tags):
        word_tag_counts = defaultdict(Counter)
        tag_counts = Counter()

        for sentence, tag_seq in zip(sentences, tags):
            for word, tag in zip(sentence, tag_seq):
                word_tag_counts[word.lower()][tag] += 1
                tag_counts[tag] += 1

        for word, tag_counter in word_tag_counts.items():
            self.word_to_tag[word] = tag_counter.most_common(1)[0][0]

        self.most_common_tag = tag_counts.most_common(1)[0][0]

    def predict(self, sentences):
        predictions = []
        for sentence in sentences:
            sentence_preds = []
            for word in sentence:
                tag = self.word_to_tag.get(word.lower(), self.most_common_tag)
                sentence_preds.append(tag)
            predictions.append(sentence_preds)
        return predictions

# Train baseline model
print("Training baseline model...")
baseline = BaselineTagger()
baseline.train(train_sentences, train_pos_tags)

# Get baseline predictions
train_baseline_preds = baseline.predict(train_sentences)
dev_baseline_preds = baseline.predict(dev_sentences)
test_baseline_preds = baseline.predict(test_sentences)


## 5. BERT Model for POS Tagging


In [ ]:
# Load BERT tokenizer
tokenizer = BertTokenizerFast.from_pretrained(BERT_MODEL_NAME)

# Function to prepare data for BERT
def prepare_data_for_bert(sentences, pos_tags, tag_to_idx, tokenizer, max_length):
    input_ids = []
    attention_masks = []
    token_type_ids = []
    labels = []
    word_ids_list = []

    for sentence, tags in zip(sentences, pos_tags):
        # Tokenize the text
        encoded = tokenizer(
            sentence,
            padding='max_length',
            truncation=True,
            max_length=max_length,
            return_tensors='tf',
            is_split_into_words=True
        )

        # Get the input IDs, attention mask, and token type IDs
        input_ids.append(encoded['input_ids'][0])
        attention_masks.append(encoded['attention_mask'][0])
        token_type_ids.append(encoded['token_type_ids'][0])

        # Get word IDs directly from the tokenizer
        word_ids = encoded.word_ids(0)

        # Initialize labels with -100 (ignored in loss calculation)
        label = [-100] * max_length

        # Assign labels based on word IDs
        for i, word_idx in enumerate(word_ids):
            if word_idx is not None and word_idx < len(tags):
                # Only set the label for the first token of each word
                if i == 0 or word_ids[i-1] != word_idx:
                    label[i] = tag_to_idx[tags[word_idx]]

        word_ids_list.append(word_ids)
        labels.append(label)

    return {
        'input_ids': np.array(input_ids),
        'attention_mask': np.array(attention_masks),
        'token_type_ids': np.array(token_type_ids)
    }, np.array(labels), word_ids_list

# Prepare data for BERT
print("Preparing data for BERT...")
train_inputs, train_labels, train_word_ids = prepare_data_for_bert(
    train_sentences, train_pos_tags, tag_to_idx, tokenizer, MAX_SEQUENCE_LENGTH
)
dev_inputs, dev_labels, dev_word_ids = prepare_data_for_bert(
    dev_sentences, dev_pos_tags, tag_to_idx, tokenizer, MAX_SEQUENCE_LENGTH
)
test_inputs, test_labels, test_word_ids = prepare_data_for_bert(
    test_sentences, test_pos_tags, tag_to_idx, tokenizer, MAX_SEQUENCE_LENGTH
)


In [ ]:
# Build BERT model for token classification
def build_bert_model(num_labels):
    # Load pre-trained BERT model
    bert = TFBertModel.from_pretrained(BERT_MODEL_NAME)

    # Define inputs
    input_ids = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='input_ids')
    attention_mask = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='attention_mask')
    token_type_ids = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='token_type_ids')

    # Get BERT embeddings
    # Use custom BertLayer to wrap the BERT model call
    bert_layer = BertLayer(bert)
    bert_outputs = bert_layer([input_ids, attention_mask, token_type_ids])

    # Use the output from BertLayer (already the last hidden state)
    sequence_output = bert_outputs

    # Add dropout
    sequence_output = Dropout(0.1)(sequence_output)

    # Add classification layer
    logits = Dense(num_labels)(sequence_output)

    # Build model
    model = Model(
        inputs=[input_ids, attention_mask, token_type_ids],
        outputs=logits
    )

    return model

# Function to evaluate predictions for sequence labeling
def evaluate_sequence_predictions(true_tags, pred_tags, tag_to_idx):
    # Flatten the predictions, excluding ignored indices (-100)
    true_flat = []
    pred_flat = []

    for i, (true_seq, pred_seq) in enumerate(zip(true_tags, pred_tags)):
        for j, (true_tag, pred_tag) in enumerate(zip(true_seq, pred_seq)):
            # Skip ignored indices
            if true_tag != -100:
                true_flat.append(true_tag)
                pred_flat.append(pred_tag)

    # Calculate metrics
    precision = precision_score(true_flat, pred_flat, average=None, zero_division=0)
    recall = recall_score(true_flat, pred_flat, average=None, zero_division=0)
    f1 = f1_score(true_flat, pred_flat, average=None, zero_division=0)

    macro_precision = precision_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_recall = recall_score(true_flat, pred_flat, average='macro', zero_division=0)
    macro_f1 = f1_score(true_flat, pred_flat, average='macro', zero_division=0)

    # Calculate PR AUC for each class
    pr_auc = []
    for i in range(len(tag_to_idx)):
        true_binary = [1 if t == i else 0 for t in true_flat]
        pred_binary = [1 if p == i else 0 for p in pred_flat]

        if sum(true_binary) > 0:  # Only if class exists in true labels
            precision_curve, recall_curve, _ = precision_recall_curve(true_binary, pred_binary)
            pr_auc.append(auc(recall_curve, precision_curve))
        else:
            pr_auc.append(0.0)

    macro_pr_auc = np.mean(pr_auc)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "pr_auc": pr_auc,
        "macro_precision": macro_precision,
        "macro_recall": macro_recall,
        "macro_f1": macro_f1,
        "macro_pr_auc": macro_pr_auc
    }

# Function to convert baseline predictions to BERT format
def convert_baseline_to_bert_format(baseline_preds, word_ids, tag_to_idx, max_length):
    bert_format_preds = []

    for preds, ids in zip(baseline_preds, word_ids):
        bert_preds = [-100] * max_length

        for i, word_id in enumerate(ids):
            if word_id is not None and word_id < len(preds):
                bert_preds[i] = tag_to_idx[preds[word_id]]

        bert_format_preds.append(bert_preds)

    return np.array(bert_format_preds)

# Convert baseline predictions to BERT format
train_baseline_bert = convert_baseline_to_bert_format(
    train_baseline_preds, train_word_ids, tag_to_idx, MAX_SEQUENCE_LENGTH
)
dev_baseline_bert = convert_baseline_to_bert_format(
    dev_baseline_preds, dev_word_ids, tag_to_idx, MAX_SEQUENCE_LENGTH
)
test_baseline_bert = convert_baseline_to_bert_format(
    test_baseline_preds, test_word_ids, tag_to_idx, MAX_SEQUENCE_LENGTH
)


## 6. Hyperparameter Tuning


In [ ]:
# # Define hyperparameter grid
# param_grid = {
#     'learning_rate': [1e-5, 2e-5, 3e-5],
#     'batch_size': [16, 32],
#     'dropout_rate': [0.1, 0.2]
# }
#
# # Function to train and evaluate a model with specific hyperparameters
# def evaluate_hyperparameters(learning_rate, batch_size, dropout_rate, epochs=3):
#     print(f"\nEvaluating model with: lr={learning_rate}, batch_size={batch_size}, dropout={dropout_rate}")
#
#     # Define inputs
#     input_ids = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='input_ids')
#     attention_mask = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='attention_mask')
#     token_type_ids = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='token_type_ids')
#
#     # Use TFBertModel as a layer in the model
#     bert_model = TFBertModel.from_pretrained(BERT_MODEL_NAME)
#
#     # Get BERT outputs
#     # Use custom BertLayer to wrap the BERT model call
#     bert_layer = BertLayer(bert_model)
#     bert_outputs = bert_layer([input_ids, attention_mask, token_type_ids])
#
#     # Use the output from BertLayer (already the last hidden state)
#     sequence_output = bert_outputs
#
#     # Add dropout
#     sequence_output = Dropout(dropout_rate)(sequence_output)
#
#     # Add classification layer
#     logits = Dense(len(tag_to_idx))(sequence_output)
#
#     # Build model
#     model = Model(
#         inputs=[input_ids, attention_mask, token_type_ids],
#         outputs=logits
#     )
#
#     # Compile model
#     optimizer = Adam(learning_rate=learning_rate)
#     model.compile(
#         optimizer=optimizer,
#         loss=masked_sparse_categorical_crossentropy,
#         metrics=['accuracy']
#     )
#
#     # Early stopping to prevent overfitting during hyperparameter search
#     early_stopping = EarlyStopping(
#         monitor='val_loss',
#         patience=2,
#         restore_best_weights=True,
#         verbose=0
#     )
#
#     # Train the model
#     history = model.fit(
#         train_inputs,
#         train_labels,
#         batch_size=batch_size,
#         epochs=epochs,
#         validation_data=(dev_inputs, dev_labels),
#         callbacks=[early_stopping],
#         verbose=0
#     )
#
#     # Evaluate on dev set
#     dev_pred = model.predict(dev_inputs, verbose=0)
#     dev_pred_classes = np.argmax(dev_pred, axis=-1)
#     dev_metrics = evaluate_sequence_predictions(dev_labels, dev_pred_classes, tag_to_idx)
#
#     # Return the evaluation metrics and the number of epochs actually trained
#     return {
#         'learning_rate': learning_rate,
#         'batch_size': batch_size,
#         'dropout_rate': dropout_rate,
#         'macro_f1': dev_metrics['macro_f1'],
#         'macro_precision': dev_metrics['macro_precision'],
#         'macro_recall': dev_metrics['macro_recall'],
#         'macro_pr_auc': dev_metrics['macro_pr_auc'],
#         'epochs_trained': len(history.history['loss']),
#         'val_loss': min(history.history['val_loss'])
#     }
#
# # Perform grid search
# print("Starting hyperparameter tuning...")
# results = []
#
# for learning_rate in param_grid['learning_rate']:
#     for batch_size in param_grid['batch_size']:
#         for dropout_rate in param_grid['dropout_rate']:
#             result = evaluate_hyperparameters(learning_rate, batch_size, dropout_rate)
#             results.append(result)
#             print(f"  F1: {result['macro_f1']:.4f}, Val Loss: {result['val_loss']:.4f}")
#
# # Convert results to DataFrame for easier analysis
# results_df = pd.DataFrame(results)
# print("\nHyperparameter tuning results:")
# print(results_df.sort_values('macro_f1', ascending=False).head())
#
# # Visualize results
# plt.figure(figsize=(15, 10))
#
# # Plot F1 score by learning rate and batch size
# plt.subplot(2, 2, 1)
# for bs in param_grid['batch_size']:
#     bs_results = results_df[results_df['batch_size'] == bs]
#     plt.plot(bs_results['learning_rate'], bs_results['macro_f1'],
#              marker='o', label=f'Batch Size {bs}')
# plt.xlabel('Learning Rate')
# plt.ylabel('Macro F1 Score')
# plt.title('F1 Score by Learning Rate and Batch Size')
# plt.legend()
# plt.grid(True)
#
# # Plot F1 score by dropout rate
# plt.subplot(2, 2, 2)
# dropout_results = results_df.groupby('dropout_rate')['macro_f1'].mean().reset_index()
# plt.bar(dropout_results['dropout_rate'].astype(str), dropout_results['macro_f1'])
# plt.xlabel('Dropout Rate')
# plt.ylabel('Average Macro F1 Score')
# plt.title('F1 Score by Dropout Rate')
# plt.grid(True, axis='y')
#
# # Plot validation loss by learning rate and batch size
# plt.subplot(2, 2, 3)
# for bs in param_grid['batch_size']:
#     bs_results = results_df[results_df['batch_size'] == bs]
#     plt.plot(bs_results['learning_rate'], bs_results['val_loss'],
#              marker='o', label=f'Batch Size {bs}')
# plt.xlabel('Learning Rate')
# plt.ylabel('Validation Loss')
# plt.title('Validation Loss by Learning Rate and Batch Size')
# plt.legend()
# plt.grid(True)
#
# # Plot number of epochs trained
# plt.subplot(2, 2, 4)
# epochs_by_params = results_df.groupby(['learning_rate', 'batch_size'])['epochs_trained'].mean().reset_index()
# plt.scatter(epochs_by_params['learning_rate'], epochs_by_params['batch_size'],
#             s=epochs_by_params['epochs_trained']*20, alpha=0.6)
# plt.xlabel('Learning Rate')
# plt.ylabel('Batch Size')
# plt.title('Average Epochs Trained')
# for i, row in epochs_by_params.iterrows():
#     plt.annotate(f"{row['epochs_trained']:.1f}",
#                  (row['learning_rate'], row['batch_size']),
#                  ha='center', va='center')
# plt.grid(True)
#
# plt.tight_layout()
# plt.show()
#
# # Get the best hyperparameters
# best_result = results_df.loc[results_df['macro_f1'].idxmax()]
# print(f"\nBest hyperparameters:")
# print(f"  Learning rate: {best_result['learning_rate']}")
# print(f"  Batch size: {best_result['batch_size']}")
# print(f"  Dropout rate: {best_result['dropout_rate']}")
# print(f"  Resulting F1 score: {best_result['macro_f1']:.4f}")
#
# # Update the hyperparameters for the final model
# LEARNING_RATE = float(best_result['learning_rate'])
# BATCH_SIZE = int(best_result['batch_size'])
# DROPOUT_RATE = float(best_result['dropout_rate'])

LEARNING_RATE = 2e-05
BATCH_SIZE = 32
DROPOUT_RATE = 0.3

## 7. Model Training and Evaluation


In [ ]:
# Build BERT model with the best hyperparameters
print(f"Building BERT model with tuned hyperparameters:")
print(f"  - Learning Rate: {LEARNING_RATE}")
print(f"  - Batch Size: {BATCH_SIZE}")
print(f"  - Dropout Rate: {DROPOUT_RATE}")

# Build model
bert = TFBertModel.from_pretrained(BERT_MODEL_NAME)

# Define inputs
input_ids = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='input_ids')
attention_mask = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='attention_mask')
token_type_ids = Input(shape=(MAX_SEQUENCE_LENGTH,), dtype=tf.int32, name='token_type_ids')

# Get BERT embeddings
# Use custom BertLayer to wrap the BERT model call
bert_layer = BertLayer(bert)
bert_outputs = bert_layer([input_ids, attention_mask, token_type_ids])

# Use the output from BertLayer (already the last hidden state)
sequence_output = bert_outputs

# Add dropout
sequence_output = Dropout(DROPOUT_RATE)(sequence_output)

# Add classification layer
logits = Dense(len(tag_to_idx))(sequence_output)

# Build model
bert_model = Model(
    inputs=[input_ids, attention_mask, token_type_ids],
    outputs=logits
)

# Compile model
optimizer = Adam(learning_rate=LEARNING_RATE)
bert_model.compile(
    optimizer=optimizer,
    loss=masked_sparse_categorical_crossentropy,
    metrics=['accuracy']
)

print("\nModel Summary:")
bert_model.summary()

# Set up callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True, verbose=1),
    ModelCheckpoint('best_bert_pos_tagger.keras', monitor='val_loss', save_best_only=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, verbose=1)
]


# Train the model
print("Training BERT model...")
history = bert_model.fit(
    train_inputs,
    train_labels,
    batch_size=BATCH_SIZE,
    epochs=EPOCHS,
    validation_data=(dev_inputs, dev_labels),
    callbacks=callbacks,
    verbose=1
)

# Plot training curves
def plot_training_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curves
    ax1.plot(history.history['loss'], label='Training Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy curves
    ax2.plot(history.history['accuracy'], label='Training Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

# Plot training curves
print("\nPlotting training curves...")
plot_training_curves(history)


In [ ]:
# Plot training curves
def plot_training_curves(history):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Loss curves
    ax1.plot(history.history['loss'], label='Training Loss')
    ax1.plot(history.history['val_loss'], label='Validation Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.set_title('Training and Validation Loss')
    ax1.legend()
    ax1.grid(True)

    # Accuracy curves
    ax2.plot(history.history['accuracy'], label='Training Accuracy')
    ax2.plot(history.history['val_accuracy'], label='Validation Accuracy')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('Accuracy')
    ax2.set_title('Training and Validation Accuracy')
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()

# Plot training curves
print("\nPlotting training curves...")
plot_training_curves(history)


In [ ]:
# Make predictions
print("Making predictions...")

# BERT predictions
train_bert_pred = bert_model.predict(train_inputs)
dev_bert_pred = bert_model.predict(dev_inputs)
test_bert_pred = bert_model.predict(test_inputs)

# Convert to class predictions
train_bert_pred_classes = np.argmax(train_bert_pred, axis=-1)
dev_bert_pred_classes = np.argmax(dev_bert_pred, axis=-1)
test_bert_pred_classes = np.argmax(test_bert_pred, axis=-1)

# Evaluate models
print("Evaluating models...")

# Evaluate BERT
train_bert_metrics = evaluate_sequence_predictions(train_labels, train_bert_pred_classes, tag_to_idx)
dev_bert_metrics = evaluate_sequence_predictions(dev_labels, dev_bert_pred_classes, tag_to_idx)
test_bert_metrics = evaluate_sequence_predictions(test_labels, test_bert_pred_classes, tag_to_idx)

# Evaluate baseline
train_baseline_metrics = evaluate_sequence_predictions(train_labels, train_baseline_bert, tag_to_idx)
dev_baseline_metrics = evaluate_sequence_predictions(dev_labels, dev_baseline_bert, tag_to_idx)
test_baseline_metrics = evaluate_sequence_predictions(test_labels, test_baseline_bert, tag_to_idx)


In [ ]:
# Print results
print("\n" + "="*80)
print("EXPERIMENTAL RESULTS")
print("="*80)

print(f"\nModel Configuration:")
print(f"- BERT Model: {BERT_MODEL_NAME}")
print(f"- Learning Rate: {LEARNING_RATE}")
print(f"- Batch Size: {BATCH_SIZE}")
print(f"- Dropout Rate: {DROPOUT_RATE}")
print(f"- Max Sequence Length: {MAX_SEQUENCE_LENGTH}")

print(f"\nBaseline Model Results:")
print("-" * 40)
print("Training Set:")
print(f"  Macro-averaged Precision: {train_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {train_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {train_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {train_baseline_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {dev_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {dev_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {dev_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {dev_baseline_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {test_baseline_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {test_baseline_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {test_baseline_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {test_baseline_metrics['macro_pr_auc']:.4f}")

print(f"\nBERT Model Results:")
print("-" * 40)
print("Training Set:")
print(f"  Macro-averaged Precision: {train_bert_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {train_bert_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {train_bert_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {train_bert_metrics['macro_pr_auc']:.4f}")

print("\nDevelopment Set:")
print(f"  Macro-averaged Precision: {dev_bert_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {dev_bert_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {dev_bert_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {dev_bert_metrics['macro_pr_auc']:.4f}")

print("\nTest Set:")
print(f"  Macro-averaged Precision: {test_bert_metrics['macro_precision']:.4f}")
print(f"  Macro-averaged Recall: {test_bert_metrics['macro_recall']:.4f}")
print(f"  Macro-averaged F1: {test_bert_metrics['macro_f1']:.4f}")
print(f"  Macro-averaged PR AUC: {test_bert_metrics['macro_pr_auc']:.4f}")


In [ ]:
print("\nPer-class Metrics for BERT Model (Train Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score\tPR AUC")
print("-" * 75)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = train_bert_metrics['precision'][i]
    recall = train_bert_metrics['recall'][i]
    f1 = train_bert_metrics['f1'][i]
    pr_auc = train_bert_metrics['pr_auc'][i]
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}\t\t{pr_auc:.4f}")

print("\nPer-class Metrics for BERT Model (Dev Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score\tPR AUC")
print("-" * 75)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = dev_bert_metrics['precision'][i]
    recall = dev_bert_metrics['recall'][i]
    f1 = dev_bert_metrics['f1'][i]
    pr_auc = dev_bert_metrics['pr_auc'][i]
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}\t\t{pr_auc:.4f}")

print("\nPer-class Metrics for BERT Model (Test Set):")
print("Class\t\tPrecision\tRecall\t\tF1 Score\tPR AUC")
print("-" * 75)
for i in range(len(tag_to_idx)):
    tag = idx_to_tag[i]
    precision = test_bert_metrics['precision'][i]
    recall = test_bert_metrics['recall'][i]
    f1 = test_bert_metrics['f1'][i]
    pr_auc = test_bert_metrics['pr_auc'][i]
    padding = "\t\t" if len(tag) < 8 else "\t"
    print(f"{tag}{padding}{precision:.4f}\t\t{recall:.4f}\t\t{f1:.4f}\t\t{pr_auc:.4f}")


In [ ]:
# Plot confusion matrix for test set
print("Plotting confusion matrix...")

# Flatten the true and predicted tags for test set
true_tags_flat = []
pred_tags_flat = []
for true_seq, pred_seq in zip(test_labels, test_bert_pred_classes):
    for true_tag, pred_tag in zip(true_seq, pred_seq):
        if true_tag != -100:  # Skip ignored indices
            true_tags_flat.append(true_tag)
            pred_tags_flat.append(pred_tag)

# Create confusion matrix
cm = confusion_matrix(true_tags_flat, pred_tags_flat)

# Normalize the confusion matrix
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

# Plot
plt.figure(figsize=(12, 10))
sns.heatmap(cm_norm, annot=True, fmt='.2f', cmap='Blues',
            xticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))],
            yticklabels=[idx_to_tag[i] for i in range(len(tag_to_idx))])
plt.title('Normalized Confusion Matrix')
plt.xlabel('Predicted Tags')
plt.ylabel('True Tags')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()


## 8. Model Comparison with Previous Approaches


In [ ]:
# Load MLP results from Assignment 3 for comparison
mlp_train_metrics = {
    "macro_precision": 0.8789,
    "macro_recall": 0.8636,
    "macro_f1": 0.8702,
    "macro_pr_auc": 0.9023
}

mlp_dev_metrics = {
    "macro_precision": 0.8265,
    "macro_recall": 0.7940,
    "macro_f1": 0.8030,
    "macro_pr_auc": 0.8430
}

mlp_test_metrics = {
    "macro_precision": 0.8335,
    "macro_recall": 0.8091,
    "macro_f1": 0.8144,
    "macro_pr_auc": 0.8540
}

# Load RNN results from Assignment 4 for comparison
rnn_train_metrics = {
    "macro_precision": 0.8827,
    "macro_recall": 0.8070,
    "macro_f1": 0.8246,
    "macro_pr_auc": 0.8460
}

rnn_dev_metrics = {
    "macro_precision": 0.8132,
    "macro_recall": 0.7470,
    "macro_f1": 0.7700,
    "macro_pr_auc": 0.8115
}

rnn_test_metrics = {
    "macro_precision": 0.8294,
    "macro_recall": 0.7727,
    "macro_f1": 0.7948,
    "macro_pr_auc": 0.8323
}

# Load CNN results from Assignment 5 for comparison
cnn_train_metrics = {
    "macro_precision": 0.9137,
    "macro_recall": 0.8930,
    "macro_f1": 0.9015,
    "macro_pr_auc": 0.9040
}

cnn_dev_metrics = {
    "macro_precision": 0.8603,
    "macro_recall": 0.8087,
    "macro_f1": 0.8231,
    "macro_pr_auc": 0.8361
}

cnn_test_metrics = {
    "macro_precision": 0.8636,
    "macro_recall": 0.8258,
    "macro_f1": 0.8307,
    "macro_pr_auc": 0.8462
}

print("\nMLP Model Results (from Assignment 3):")
print("Test Set:")
print(f"  Macro-averaged F1: {mlp_test_metrics['macro_f1']:.4f}")

print("\nRNN Model Results (from Assignment 4):")
print("Test Set:")
print(f"  Macro-averaged F1: {rnn_test_metrics['macro_f1']:.4f}")

print("\nCNN Model Results (from Assignment 5):")
print("Test Set:")
print(f"  Macro-averaged F1: {cnn_test_metrics['macro_f1']:.4f}")

print("\nBERT Model Results:")
print("Test Set:")
print(f"  Macro-averaged F1: {test_bert_metrics['macro_f1']:.4f}")

# Create a bar chart to compare model performance
def plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, bert_metrics, metric_name):
    models = ['Baseline', 'MLP', 'RNN', 'CNN', 'BERT']
    train_values = [
        baseline_metrics['train'][metric_name],
        mlp_metrics['train'][metric_name],
        rnn_metrics['train'][metric_name],
        cnn_metrics['train'][metric_name],
        bert_metrics['train'][metric_name]
    ]
    dev_values = [
        baseline_metrics['dev'][metric_name],
        mlp_metrics['dev'][metric_name],
        rnn_metrics['dev'][metric_name],
        cnn_metrics['dev'][metric_name],
        bert_metrics['dev'][metric_name]
    ]
    test_values = [
        baseline_metrics['test'][metric_name],
        mlp_metrics['test'][metric_name],
        rnn_metrics['test'][metric_name],
        cnn_metrics['test'][metric_name],
        bert_metrics['test'][metric_name]
    ]

    x = np.arange(len(models))
    width = 0.25

    fig, ax = plt.subplots(figsize=(12, 6))
    ax.bar(x - width, train_values, width, label='Train')
    ax.bar(x, dev_values, width, label='Dev')
    ax.bar(x + width, test_values, width, label='Test')

    ax.set_ylabel(f'{metric_name.replace("macro_", "Macro-averaged ")}')
    ax.set_title(f'Model Comparison - {metric_name.replace("macro_", "Macro-averaged ")}')
    ax.set_xticks(x)
    ax.set_xticklabels(models)
    ax.legend()
    ax.grid(True, axis='y')

    # Add value labels on top of bars
    for i, v in enumerate(train_values):
        ax.text(i - width, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    for i, v in enumerate(dev_values):
        ax.text(i, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=8)
    for i, v in enumerate(test_values):
        ax.text(i + width, v + 0.01, f'{v:.4f}', ha='center', va='bottom', fontsize=8)

    plt.tight_layout()
    plt.show()

# Organize metrics for comparison
baseline_metrics = {
    'train': train_baseline_metrics,
    'dev': dev_baseline_metrics,
    'test': test_baseline_metrics
}

mlp_metrics = {
    'train': mlp_train_metrics,
    'dev': mlp_dev_metrics,
    'test': mlp_test_metrics
}

rnn_metrics = {
    'train': rnn_train_metrics,
    'dev': rnn_dev_metrics,
    'test': rnn_test_metrics
}

cnn_metrics = {
    'train': cnn_train_metrics,
    'dev': cnn_dev_metrics,
    'test': cnn_test_metrics
}

bert_metrics = {
    'train': train_bert_metrics,
    'dev': dev_bert_metrics,
    'test': test_bert_metrics
}

# Plot comparisons for different metrics
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, bert_metrics, 'macro_f1')
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, bert_metrics, 'macro_precision')
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, bert_metrics, 'macro_recall')
plot_model_comparison(baseline_metrics, mlp_metrics, rnn_metrics, cnn_metrics, bert_metrics, 'macro_pr_auc')
